# 08 — CLIP embedding, FAISS index, retrieval quality

**Project:** RAOD++ · **Phase:** F6 · **Protocol:** v1.2

Turns crops into the retrieval stage:

1. embed the gallery (47,082 crops) and every query set (10 × ~127k crops) with CLIP ViT-B/32,
2. build a FAISS index over the gallery (cosine similarity),
3. retrieve the k nearest gallery neighbours for every query,
4. **measure whether retrieval is semantically meaningful** — class-consistency@k,
   similarity distributions, and the UMAP projection for figure F10.

Step 4 decides whether the VisDrone test-dev gallery is good enough or whether the COCO
pool should be promoted to primary (protocol v1.2 A7 keeps both on the table).

**Runtime:** gallery ~2 min · queries ~25–40 min · retrieval ~10 min.
**Disk:** ~1.4 GB of embeddings.


## [Cell 3] Dependency and weight availability check
CLIP weights are fetched from an external host, and this network blocks several CDNs.
This cell fails fast and loudly rather than after 40 minutes of embedding.

In [1]:
# ===== Cell 3 =====
import subprocess, sys

try:
    import open_clip
    print("open_clip:", open_clip.__version__ if hasattr(open_clip,'__version__') else "ok")
except ImportError:
    print("installing open_clip_torch ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)
    import open_clip

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

try:
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai")
    print("CLIP ViT-B/32 weights: OK")
    del model
    torch.cuda.empty_cache()
except Exception as e:
    print("CLIP WEIGHT DOWNLOAD FAILED:", type(e).__name__, e)
    print("\nIf this is a network block, download the checkpoint manually in Windows and")
    print("place it in ~/.cache/huggingface or ~/.cache/clip, then re-run this cell.")
    raise


/home/aakkaya/RAODpp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


open_clip: 3.3.0
torch: 2.13.0+cu130 | cuda: True


/home/aakkaya/RAODpp/.venv/lib/python3.14/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP ViT-B/32 weights: OK


## [Cell 5] Config

In [ ]:
# ===== Cell 5 =====
from pathlib import Path
import numpy as np, pandas as pd

CONFIG = {
    "PROJECT_ROOT": Path.home() / "RAODpp",
    "SEEDS": [42, 123, 2024, 7, 3407, 11, 55, 101, 777, 12345],
    "EXP": "E0", "DATASET": "visdrone",
    "CLIP_ARCH": "ViT-B-32-quickgelu",
    "CLIP_PRETRAIN": "openai",
    "BATCH": 512,
    "WORKERS": 8,
    "FP16": True,
    "K": 5,                    # neighbours per query (protocol v1.0)
    "K_EVAL": [1, 3, 5, 10],   # for the consistency curve
    "UMAP_SAMPLE": 30000,
}

ROOT = CONFIG["PROJECT_ROOT"]
OUT = ROOT/"output"; FIGDATA = OUT/"figdata"
CROPS = ROOT/"dataset"/"crops"
EMB = ROOT/"dataset"/"embeddings"; EMB.mkdir(parents=True, exist_ok=True)

gal_man = pd.read_csv(CROPS/"manifest_gallery_visdrone.csv")
print("gallery crops:", len(gal_man))
print("query sets   :", len(list(CROPS.glob("manifest_queries_*.csv"))))
print("embeddings ->", EMB)


gallery crops: 47082
query sets   : 10
embeddings -> /home/aakkaya/RAODpp/dataset/embeddings


## [Cell 7] Embedding function
Crops are already 224×224, so the transform is just tensor conversion plus CLIP's
normalisation. Embeddings are L2-normalised on write, which makes inner product equal to
cosine similarity and lets FAISS use a flat IP index.

In [ ]:
# ===== Cell 7 (revised: fork start method + quickgelu) =====
import torch, torch.multiprocessing as mp
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import open_clip, time
import torch.multiprocessing as mp
mp.set_start_method("fork", force=True)

# Python 3.14 defaults to forkserver on Linux, which cannot pickle classes
# defined in a notebook. fork inherits them.
try:
    mp.set_start_method("fork", force=True)
    print("mp start method:", mp.get_start_method())
except RuntimeError as e:
    print("could not set fork:", e)

device = "cuda"
model, _, preprocess = open_clip.create_model_and_transforms(
    CONFIG["CLIP_ARCH"], pretrained=CONFIG["CLIP_PRETRAIN"])
model = model.to(device).eval()
if CONFIG["FP16"]:
    model = model.half()

class CropSet(Dataset):
    def __init__(self, files, root):
        self.files = files; self.root = Path(root)
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        return preprocess(Image.open(self.root/self.files[i]).convert("RGB"))

@torch.no_grad()
def embed(files, root, out_npy, tag):
    out_npy = Path(out_npy)
    if out_npy.exists():
        a = np.load(out_npy, mmap_mode="r")
        print(f"  {tag}: cached {a.shape}")
        return out_npy

    def make_loader(nw):
        return DataLoader(CropSet(files, root), batch_size=CONFIG["BATCH"],
                          num_workers=nw, pin_memory=True)
    try:
        dl = make_loader(CONFIG["WORKERS"])
        it = iter(dl); first = next(it)
    except Exception as e:
        print(f"  [{tag}] workers failed ({type(e).__name__}), falling back to 0")
        dl = make_loader(0); it = iter(dl); first = next(it)

    chunks, t0, done = [], time.time(), 0
    for x in [first] + list(it):
        x = x.to(device, non_blocking=True)
        if CONFIG["FP16"]: x = x.half()
        f = model.encode_image(x)
        f = f / f.norm(dim=-1, keepdim=True)
        chunks.append(f.cpu().numpy().astype(np.float16))
        done += len(x)
        print(f"    {tag}: {done}/{len(files)}  {done/(time.time()-t0):.0f} img/s", end="\r")
    arr = np.concatenate(chunks, axis=0)
    np.save(out_npy, arr)
    print(f"\n  {tag}: {arr.shape} in {(time.time()-t0)/60:.1f} min")
    return out_npy

print("model ready:", CONFIG["CLIP_ARCH"], CONFIG["CLIP_PRETRAIN"])

mp start method: fork
model ready: ViT-B-32-quickgelu openai


## [Cell 9] Embed the gallery

In [4]:
# ===== Cell 9 =====
gal_npy = embed(gal_man.crop.tolist(), CROPS/"gallery_visdrone",
                EMB/"gallery_visdrone.npy", "gallery")
G = np.load(gal_npy).astype(np.float32)
print("gallery embeddings:", G.shape)
print("norm check (should be ~1.0):", float(np.linalg.norm(G[0])))


    gallery: 47082/47082  1251 img/s
  gallery: (47082, 512) in 0.6 min
gallery embeddings: (47082, 512)
norm check (should be ~1.0): 1.0002278089523315


## [Cell 11] Build the FAISS index
The gallery has 47k vectors, so an exact flat index is both feasible and preferable — no
approximation error to explain, and the complexity claim in the manuscript becomes a
simple exact-search statement instead of the IVF formula used in the v1 paper.

In [5]:
# ===== Cell 11 =====
try:
    import faiss
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=True)
    import faiss

index = faiss.IndexFlatIP(G.shape[1])      # inner product on L2-normalised = cosine
index.add(G)
faiss.write_index(index, str(EMB/"faiss_gallery_visdrone.index"))
print("index size:", index.ntotal, "| dim:", index.d, "| exact search (IndexFlatIP)")


index size: 47082 | dim: 512 | exact search (IndexFlatIP)


## [Cell 13] Embed queries and retrieve neighbours
One pass per seed. Retrieval results are stored as a compact table: query row index,
neighbour gallery index, rank and cosine similarity.

In [ ]:
# ===== Cell 13 =====
KMAX = max(CONFIG["K_EVAL"])
ret_summary = []

for seed in CONFIG["SEEDS"]:
    rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    man = pd.read_csv(CROPS/f"manifest_queries_{rid}.csv")
    npy = embed(man.crop.tolist(), CROPS/f"queries_{rid}",
                EMB/f"queries_{rid}.npy", f"q_s{seed}")

    ret_path = EMB/f"retrieval_{rid}.npz"
    if ret_path.exists():
        z = np.load(ret_path); D, I = z["D"], z["I"]
        print(f"  {rid}: retrieval cached {I.shape}")
    else:
        Q = np.load(npy).astype(np.float32)
        D, I = index.search(Q, KMAX)
        np.savez_compressed(ret_path, D=D.astype(np.float16), I=I.astype(np.int32))
        print(f"  {rid}: retrieved {I.shape}, mean top1 sim = {D[:,0].mean():.4f}")
        del Q
    ret_summary.append({"seed": seed, "n_queries": len(man),
                        "mean_sim_top1": float(np.mean(D[:,0])),
                        "mean_sim_topk": float(np.mean(D[:,:CONFIG["K"]]))})

rs = pd.DataFrame(ret_summary)
rs.to_csv(FIGDATA/"retrieval_similarity_summary.csv", index=False)
print(rs.round(4).to_string(index=False))


## [Cell 15] Retrieval quality — class-consistency@k

The honest version of the v1 manuscript's unsupported "92% semantic similarity" claim.

Both the query label and the gallery label are *predicted*, not ground truth: queries come
from the uncertainty band (precision 0.5–0.9 by construction) and gallery labels come from
a detector used as a region proposer. Consistency is therefore a lower bound on semantic
agreement, and the manuscript says so rather than presenting it as an accuracy.

A random-match baseline is computed from the gallery class distribution so the number can
be read against something.

In [ ]:
# ===== Cell 15 =====
gal_cls = gal_man.class_id.values
p_gal = gal_man.class_id.value_counts(normalize=True).sort_index()
random_baseline = float((p_gal**2).sum())     # chance of a class match under sampling

rows = []
for seed in CONFIG["SEEDS"]:
    rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    man = pd.read_csv(CROPS/f"manifest_queries_{rid}.csv")
    z = np.load(EMB/f"retrieval_{rid}.npz"); I = z["I"]
    q_cls = man.class_id.values
    for k in CONFIG["K_EVAL"]:
        nb = gal_cls[I[:, :k]]
        any_match = (nb == q_cls[:, None]).any(axis=1).mean()
        all_match = (nb == q_cls[:, None]).mean()
        rows.append({"seed": seed, "k": k,
                     "consistency_any": float(any_match),
                     "consistency_mean": float(all_match)})

cons = pd.DataFrame(rows)
cons.to_csv(FIGDATA/"retrieval_quality.csv", index=False)
tab = cons.groupby("k")[["consistency_any","consistency_mean"]].agg(["mean","std"]).round(4)
print("class-consistency@k over 10 seeds")
print(tab.to_string())
print(f"\nrandom-match baseline (gallery prior): {random_baseline:.4f}")

# per-class breakdown at k=5, reference seed
rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s42"
man = pd.read_csv(CROPS/f"manifest_queries_{rid}.csv")
I = np.load(EMB/f"retrieval_{rid}.npz")["I"]
nb = gal_cls[I[:, :CONFIG["K"]]]
man["hit"] = (nb == man.class_id.values[:, None]).any(axis=1)
per_cls = (man.groupby("class_name")
           .agg(queries=("hit","size"), consistency=("hit","mean")).round(4)
           .sort_values("consistency"))
per_cls["gallery_crops"] = per_cls.index.map(gal_man.class_name.value_counts())
per_cls.to_csv(FIGDATA/"retrieval_quality_per_class.csv")
print()
print(per_cls.to_string())


## [Cell 17] UMAP projection (figure F10)
Only coordinates are stored; the figure itself is drawn in F9 from this CSV.

In [ ]:
# ===== Cell 17 =====
try:
    import umap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=True)
    import umap

rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s42"
man = pd.read_csv(CROPS/f"manifest_queries_{rid}.csv")
Q = np.load(EMB/f"queries_{rid}.npy").astype(np.float32)

rng = np.random.default_rng(42)
idx = rng.choice(len(Q), size=min(CONFIG["UMAP_SAMPLE"], len(Q)), replace=False)
coords = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine",
                   random_state=42).fit_transform(Q[idx])

pd.DataFrame({"x": coords[:,0], "y": coords[:,1],
              "class_id": man.class_id.values[idx],
              "class_name": man.class_name.values[idx],
              "conf": man.conf.values[idx],
              "box_area": man.box_area.values[idx]}
             ).to_csv(FIGDATA/"umap_coords.csv", index=False)
print("umap_coords.csv:", len(idx), "points")


**Decision point.** Report back: the class-consistency table with the random baseline,
and the per-class breakdown. If consistency on the rare classes collapses, the COCO pool is
promoted to primary gallery before E1 starts (protocol v1.2 A7).

**Next:** `09_upaste_enrichment.ipynb` — uncertainty-driven placement, cross-modal
pseudo-label validation and the enriched training sets for E1–E5.